In [1]:
!pip install -q gradio matplotlib numpy

import gradio as gr
import matplotlib.pyplot as plt
import numpy as np

AVAILABLE_AIRFOILS = ["NACA 0012", "NACA 2412", "NACA 4412", "NACA 2415", "NACA 6412"]


def parse_naca_4digit(code: str):
    """Parses a 4-digit NACA string like 'NACA 2412' into (m, p, t) fractions."""
    digits = "".join(ch for ch in code if ch.isdigit())
    if len(digits) != 4:
        digits = "2412"
    m = int(digits[0]) / 100.0
    p = int(digits[1]) / 10.0
    t = int(digits[2:]) / 100.0
    return m, p, t


def generate_naca_airfoil(code: str, n_pts: int = 120):
    """Generates upper and lower (x/c, y/c) coordinates for a 4-digit NACA airfoil."""
    m, p, t = parse_naca_4digit(code)
    beta = np.linspace(0.0, np.pi, n_pts)
    xc = 0.5 * (1.0 - np.cos(beta))

    yt = 5.0 * t * (
        0.2969 * np.sqrt(xc)
        - 0.1260 * xc
        - 0.3516 * (xc ** 2)
        + 0.2843 * (xc ** 3)
        - 0.1036 * (xc ** 4)
    )

    yc = np.zeros_like(xc)
    dyc_dx = np.zeros_like(xc)

    if m > 0.0 and p > 0.0:
        mask_f = xc < p
        mask_a = ~mask_f
        yc[mask_f] = (m / (p ** 2)) * (2.0 * p * xc[mask_f] - xc[mask_f] ** 2)
        dyc_dx[mask_f] = (2.0 * m / (p ** 2)) * (p - xc[mask_f])

        yc[mask_a] = (m / ((1.0 - p) ** 2)) * ((1.0 - 2.0 * p) + 2.0 * p * xc[mask_a] - xc[mask_a] ** 2)
        dyc_dx[mask_a] = (2.0 * m / ((1.0 - p) ** 2)) * (p - xc[mask_a])

    theta = np.arctan(dyc_dx)
    xu = xc - yt * np.sin(theta)
    yu = yc + yt * np.cos(theta)
    xl = xc + yt * np.sin(theta)
    yl = yc - yt * np.cos(theta)

    return xc, yc, xu, yu, xl, yl, m, t


def simulate_airfoil_tradeoff(
    selected_airfoils, alpha_min_deg, alpha_max_deg,
    aspect_ratio, oswald_e, wing_area_m2, mass_kg, velocity_ms, start_alt_m
):
    if not selected_airfoils:
        selected_airfoils = ["NACA 0012", "NACA 2412", "NACA 4412"]

    rho = 1.225
    g = 9.80665
    weight_n = max(mass_kg, 0.2) * g
    s_ref = max(wing_area_m2, 0.05)

    alpha_deg = np.linspace(alpha_min_deg, alpha_max_deg, 200)
    alpha_rad = np.radians(alpha_deg)
    v_sweep = np.linspace(8.0, 65.0, 180)

    re_factor = np.clip((max(velocity_ms, 5.0) / 25.0) ** (-0.12), 0.78, 1.25)
    palette = ["#38bdf8", "#34d399", "#fbbf24", "#f472b6", "#a78bfa"]

    # 3x2 Single Visual Trade-Off Report
    fig, axes = plt.subplots(3, 2, figsize=(13.5, 11.2), dpi=130)
    fig.patch.set_facecolor("#0f172a")
    for ax in axes.flat:
        ax.set_facecolor("#0f172a")
        ax.tick_params(colors="#94a3b8", labelsize=8)
        for spine in ax.spines.values():
            spine.set_color("#334155")

    ax_geom, ax_cl = axes[0, 0], axes[0, 1]
    ax_ld, ax_polar = axes[1, 0], axes[1, 1]
    ax_drag_v, ax_glide = axes[2, 0], axes[2, 1]

    report_lines = []
    k_ind = 1.0 / (np.pi * max(aspect_ratio, 1.0) * max(oswald_e, 0.3))

    for idx, code in enumerate(selected_airfoils):
        color = palette[idx % len(palette)]
        xc, yc, xu, yu, xl, yl, m, t = generate_naca_airfoil(code)

        # 1. Airfoil Geometry
        ax_geom.plot(xu, yu, color=color, lw=1.8, label=f"{code} (t={t*100:.0f}%, m={m*100:.0f}%)")
        ax_geom.plot(xl, yl, color=color, lw=1.8)
        ax_geom.plot(xc, yc, color=color, linestyle=":", lw=0.9, alpha=0.75)

        # Finite-wing lift slope
        a0 = 2.0 * np.pi * 0.92
        cl_slope = a0 / (1.0 + a0 * k_ind)

        alpha_0l_deg = -45.0 * m
        alpha_eff_rad = np.radians(alpha_deg - alpha_0l_deg)

        stall_deg = np.clip(15.5 - 12.0 * max(m - 0.02, 0.0) + 18.0 * max(t - 0.12, 0.0), 10.0, 20.0)
        stall_rad = np.radians(stall_deg)

        cl_lin = cl_slope * alpha_eff_rad
        excess = np.maximum(0.0, np.abs(alpha_rad) - stall_rad)
        stall_factor = 1.0 / (1.0 + 3.5 * (excess ** 1.6))
        cl_total = cl_lin * stall_factor + 0.35 * np.sin(2.0 * alpha_rad) * (1.0 - stall_factor)

        cd0 = (0.0052 + 0.022 * t + 0.04 * (m ** 2)) * re_factor
        cd_induced = k_ind * (cl_total ** 2)
        stall_drag = 0.25 * (excess ** 1.5)
        cd_total = cd0 + cd_induced + stall_drag

        ld_ratio = cl_total / np.maximum(cd_total, 1e-4)

        # 2, 3, 4. Core Aerodynamic Curves
        ax_cl.plot(alpha_deg, cl_total, color=color, lw=2.0, label=code)
        ax_ld.plot(alpha_deg, ld_ratio, color=color, lw=2.0, label=code)
        ax_polar.plot(cd_total, cl_total, color=color, lw=2.0, label=code)

        # 5. Level-Flight Drag vs. Airspeed (L = W -> CL_req = W / (0.5 * rho * V^2 * S))
        q_v = 0.5 * rho * (v_sweep ** 2)
        cl_req_v = weight_n / (q_v * s_ref)
        d_par_v = q_v * s_ref * cd0
        d_ind_v = q_v * s_ref * k_ind * (cl_req_v ** 2)
        d_tot_v = d_par_v + d_ind_v

        v_best_ld = float(v_sweep[int(np.argmin(d_tot_v))])
        min_drag_n = float(np.min(d_tot_v))

        ax_drag_v.plot(v_sweep, d_tot_v, color=color, lw=2.0, label=f"{code} (V_bg={v_best_ld:.1f} m/s)")
        ax_drag_v.scatter([v_best_ld], [min_drag_n], color=color, s=35, zorder=5)

        # 6. Glide Range from start_alt_m (Range = h0 * (L/D) for positive L/D)
        glide_range_km = (max(start_alt_m, 10.0) * np.maximum(ld_ratio, 0.0)) / 1000.0
        ax_glide.plot(alpha_deg, glide_range_km, color=color, lw=2.0, label=code)

        max_cl = float(np.max(cl_total))
        best_ld = float(np.max(ld_ratio))
        best_ld_aoa = float(alpha_deg[int(np.argmax(ld_ratio))])
        max_glide_km = float(np.max(glide_range_km))
        v_stall = float(np.sqrt((2.0 * weight_n) / (rho * s_ref * max(max_cl, 0.2))))

        report_lines.append(
            f"- **{code}:** Peak $C_L \\approx {max_cl:.2f}$ ($V_{{\\text{{stall}}}} \\approx {v_stall:.1f}$ m/s), "
            f"Best $L/D \\approx {best_ld:.1f}$ at $\\alpha \\approx {best_ld_aoa:.1f}^\\circ$, "
            f"Best-Glide Speed $V_{{bg}} \\approx {v_best_ld:.1f}$ m/s, Max Glide Range $\\approx {max_glide_km:.2f}$ km."
        )

    # Styling Panel 1: Geometry
    ax_geom.axhline(0.0, color="#64748b", linestyle="--", lw=0.8)
    ax_geom.set_aspect("equal", adjustable="box")
    ax_geom.set_xlim(-0.05, 1.05)
    ax_geom.set_ylim(-0.35, 0.35)
    ax_geom.set_title("1. Airfoil Cross-Sections (x/c)", color="#f1f5f9", fontsize=10, pad=6)
    ax_geom.set_xlabel("Chordwise Position, x/c", color="#cbd5e1", fontsize=8.5)
    ax_geom.set_ylabel("y/c", color="#cbd5e1", fontsize=8.5)
    ax_geom.legend(loc="upper right", fontsize=7, facecolor="#1e293b", edgecolor="#475569", labelcolor="#f8fafc")

    # Styling Panel 2: CL vs AoA
    ax_cl.axhline(0.0, color="#64748b", linestyle=":", lw=0.8)
    ax_cl.axvline(0.0, color="#64748b", linestyle=":", lw=0.8)
    ax_cl.set_title(f"2. Lift Curve — CL vs. AoA (AR={aspect_ratio:.1f}, e={oswald_e:.2f})", color="#f1f5f9", fontsize=10, pad=6)
    ax_cl.set_xlabel("Angle of Attack, α (deg)", color="#cbd5e1", fontsize=8.5)
    ax_cl.set_ylabel("Lift Coefficient, CL", color="#cbd5e1", fontsize=8.5)
    ax_cl.legend(loc="upper left", fontsize=7.5, facecolor="#1e293b", edgecolor="#475569", labelcolor="#f8fafc")

    # Styling Panel 3: L/D vs AoA
    ax_ld.axhline(0.0, color="#64748b", linestyle=":", lw=0.8)
    ax_ld.set_title("3. Aerodynamic Efficiency — L/D vs. AoA", color="#f1f5f9", fontsize=10, pad=6)
    ax_ld.set_xlabel("Angle of Attack, α (deg)", color="#cbd5e1", fontsize=8.5)
    ax_ld.set_ylabel("Lift-to-Drag Ratio, L/D", color="#cbd5e1", fontsize=8.5)
    ax_ld.legend(loc="upper right", fontsize=7.5, facecolor="#1e293b", edgecolor="#475569", labelcolor="#f8fafc")

    # Styling Panel 4: Drag Polar
    ax_polar.set_title("4. Drag Polar — CL vs. CD", color="#f1f5f9", fontsize=10, pad=6)
    ax_polar.set_xlabel("Total Drag Coefficient, CD", color="#cbd5e1", fontsize=8.5)
    ax_polar.set_ylabel("Lift Coefficient, CL", color="#cbd5e1", fontsize=8.5)
    ax_polar.legend(loc="lower right", fontsize=7.5, facecolor="#1e293b", edgecolor="#475569", labelcolor="#f8fafc")

    # Styling Panel 5: Level-Flight Drag vs Airspeed
    ax_drag_v.axvline(velocity_ms, color="#f8fafc", linestyle="--", lw=1.0, alpha=0.75, label=f"Design Cruise ({velocity_ms:.0f} m/s)")
    ax_drag_v.set_title(f"5. Level-Flight Total Drag vs. Airspeed (m={mass_kg:.1f} kg, S={s_ref:.2f} m²)", color="#f1f5f9", fontsize=10, pad=6)
    ax_drag_v.set_xlabel("Airspeed, V (m/s)", color="#cbd5e1", fontsize=8.5)
    ax_drag_v.set_ylabel("Total Drag Force, D (N)", color="#cbd5e1", fontsize=8.5)
    ax_drag_v.set_ylim(bottom=0.0, top=max(weight_n * 0.45, 5.0))
    ax_drag_v.legend(loc="upper right", fontsize=7, facecolor="#1e293b", edgecolor="#475569", labelcolor="#f8fafc")

    # Styling Panel 6: Glide Range vs AoA
    ax_glide.set_title(f"6. Still-Air Glide Range from h₀ = {start_alt_m:.0f} m", color="#f1f5f9", fontsize=10, pad=6)
    ax_glide.set_xlabel("Angle of Attack, α (deg)", color="#cbd5e1", fontsize=8.5)
    ax_glide.set_ylabel("Glide Distance (km)", color="#cbd5e1", fontsize=8.5)
    ax_glide.legend(loc="upper right", fontsize=7.5, facecolor="#1e293b", edgecolor="#475569", labelcolor="#f8fafc")

    fig.suptitle(
        f"Wing Profile Lift/Drag, Cruise Speed & Glide Trade-Off Report  |  AR={aspect_ratio:.1f}, S={s_ref:.2f} m², m={mass_kg:.1f} kg, h₀={start_alt_m:.0f} m",
        color="#f8fafc", fontsize=11.5, fontweight="bold", y=0.985
    )
    plt.tight_layout(rect=[0, 0, 1, 0.965])

    tradeoff_md = (
        "### 📊 Design Trade-Off, Cruise & Glide Summary\n"
        + "\n".join(report_lines)
        + "\n\n**Key Engineering Trade-Offs:**\n"
        "- **High Camber (e.g., NACA 4412 / 6412):** Lowers stall speed ($V_{\\text{stall}}$) and increases max glide distance from $h_0$, "
        "but increases parasitic drag at high cruise airspeeds where induced drag drops.\n"
        "- **Symmetric / Low Camber (e.g., NACA 0012 / 2412):** Better suited for higher-speed cruise and symmetric/aerobatic maneuvering, "
        "at the cost of higher stall speed and shorter glide range.\n"
        "- **Best-Glide Speed ($V_{bg}$):** Occurs where induced drag equals parasitic drag ($D_{\\text{induced}} = D_{\\text{parasitic}}$), maximizing $L/D$ and glide range."
    )

    return fig, tradeoff_md


with gr.Blocks() as demo:
    gr.Markdown("## ✈️ Wing Profile Lift/Drag, Cruise & Glide Visual Trade-Off Tool")

    with gr.Row():
        with gr.Column(scale=1):
            selected_airfoils = gr.CheckboxGroup(
                choices=AVAILABLE_AIRFOILS,
                value=["NACA 0012", "NACA 2412", "NACA 4412"],
                label="Select Wing / Airfoil Profiles"
            )
            alpha_min_deg = gr.Slider(-10.0, 4.0, value=-4.0, step=1.0, label="Min Angle of Attack (deg)")
            alpha_max_deg = gr.Slider(8.0, 26.0, value=18.0, step=1.0, label="Max Angle of Attack (deg)")
            aspect_ratio = gr.Slider(3.0, 16.0, value=8.0, step=0.5, label="Wing Aspect Ratio (AR)")
            oswald_e = gr.Slider(0.55, 0.95, value=0.80, step=0.02, label="Oswald Efficiency Factor (e)")
            wing_area_m2 = gr.Slider(0.10, 2.50, value=0.45, step=0.05, label="Wing Reference Area, S (m²)")
            mass_kg = gr.Slider(0.5, 25.0, value=3.5, step=0.5, label="Aircraft Mass, m (kg)")
            velocity_ms = gr.Slider(10.0, 65.0, value=22.0, step=1.0, label="Design Cruise Airspeed, V (m/s)")
            start_alt_m = gr.Slider(50.0, 2000.0, value=300.0, step=50.0, label="Initial Glide Altitude, h₀ (m)")

            run_btn = gr.Button("Generate Visual Trade-Off Report", variant="primary")

        with gr.Column(scale=2):
            report_plot = gr.Plot(label="6-Panel Visual Trade-Off Report")
            report_text = gr.Markdown()

    inputs = [
        selected_airfoils, alpha_min_deg, alpha_max_deg,
        aspect_ratio, oswald_e, wing_area_m2, mass_kg, velocity_ms, start_alt_m
    ]
    run_btn.click(fn=simulate_airfoil_tradeoff, inputs=inputs, outputs=[report_plot, report_text])
    demo.load(fn=simulate_airfoil_tradeoff, inputs=inputs, outputs=[report_plot, report_text])

demo.launch(share=True, theme=gr.themes.Soft())

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://82924090406e6419f6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [2]:
!pip install -q gradio matplotlib numpy

import gradio as gr
import matplotlib.pyplot as plt
import numpy as np

AVAILABLE_AIRFOILS = ["NACA 0012", "NACA 2412", "NACA 4412", "NACA 2415", "NACA 6412"]


def parse_naca_4digit(code: str):
    """Parses a 4-digit NACA string like 'NACA 2412' into (m, p, t) fractions."""
    digits = "".join(ch for ch in code if ch.isdigit())
    if len(digits) != 4:
        digits = "2412"
    m = int(digits[0]) / 100.0
    p = int(digits[1]) / 10.0
    t = int(digits[2:]) / 100.0
    return m, p, t


def generate_naca_airfoil(code: str, n_pts: int = 120):
    """Generates upper and lower (x/c, y/c) coordinates for a 4-digit NACA airfoil."""
    m, p, t = parse_naca_4digit(code)
    beta = np.linspace(0.0, np.pi, n_pts)
    xc = 0.5 * (1.0 - np.cos(beta))

    yt = 5.0 * t * (
        0.2969 * np.sqrt(xc)
        - 0.1260 * xc
        - 0.3516 * (xc ** 2)
        + 0.2843 * (xc ** 3)
        - 0.1036 * (xc ** 4)
    )

    yc = np.zeros_like(xc)
    dyc_dx = np.zeros_like(xc)

    if m > 0.0 and p > 0.0:
        mask_f = xc < p
        mask_a = ~mask_f
        yc[mask_f] = (m / (p ** 2)) * (2.0 * p * xc[mask_f] - xc[mask_f] ** 2)
        dyc_dx[mask_f] = (2.0 * m / (p ** 2)) * (p - xc[mask_f])

        yc[mask_a] = (m / ((1.0 - p) ** 2)) * ((1.0 - 2.0 * p) + 2.0 * p * xc[mask_a] - xc[mask_a] ** 2)
        dyc_dx[mask_a] = (2.0 * m / ((1.0 - p) ** 2)) * (p - xc[mask_a])

    theta = np.arctan(dyc_dx)
    xu = xc - yt * np.sin(theta)
    yu = yc + yt * np.cos(theta)
    xl = xc + yt * np.sin(theta)
    yl = yc - yt * np.cos(theta)

    return xc, yc, xu, yu, xl, yl, m, t


def simulate_airfoil_tradeoff(
    selected_airfoils, alpha_min_deg, alpha_max_deg,
    aspect_ratio, oswald_e, wing_area_m2, mass_kg, velocity_ms, start_alt_m
):
    if not selected_airfoils:
        selected_airfoils = ["NACA 0012", "NACA 2412", "NACA 4412"]

    rho = 1.225
    g = 9.80665
    weight_n = max(mass_kg, 0.2) * g
    s_ref = max(wing_area_m2, 0.05)

    alpha_deg = np.linspace(alpha_min_deg, alpha_max_deg, 200)
    alpha_rad = np.radians(alpha_deg)
    v_sweep = np.linspace(8.0, 65.0, 180)

    re_factor = np.clip((max(velocity_ms, 5.0) / 25.0) ** (-0.12), 0.78, 1.25)
    palette = ["#38bdf8", "#34d399", "#fbbf24", "#f472b6", "#a78bfa"]

    # 3x2 Single Visual Trade-Off Report
    fig, axes = plt.subplots(3, 2, figsize=(13.5, 11.2), dpi=130)
    fig.patch.set_facecolor("#0f172a")
    for ax in axes.flat:
        ax.set_facecolor("#0f172a")
        ax.tick_params(colors="#94a3b8", labelsize=8)
        for spine in ax.spines.values():
            spine.set_color("#334155")

    ax_geom, ax_cl = axes[0, 0], axes[0, 1]
    ax_ld, ax_polar = axes[1, 0], axes[1, 1]
    ax_drag_v, ax_glide = axes[2, 0], axes[2, 1]

    report_lines = []
    k_ind = 1.0 / (np.pi * max(aspect_ratio, 1.0) * max(oswald_e, 0.3))

    for idx, code in enumerate(selected_airfoils):
        color = palette[idx % len(palette)]
        xc, yc, xu, yu, xl, yl, m, t = generate_naca_airfoil(code)

        # 1. Airfoil Geometry
        ax_geom.plot(xu, yu, color=color, lw=1.8, label=f"{code} (t={t*100:.0f}%, m={m*100:.0f}%)")
        ax_geom.plot(xl, yl, color=color, lw=1.8)
        ax_geom.plot(xc, yc, color=color, linestyle=":", lw=0.9, alpha=0.75)

        # Finite-wing lift slope
        a0 = 2.0 * np.pi * 0.92
        cl_slope = a0 / (1.0 + a0 * k_ind)

        alpha_0l_deg = -45.0 * m
        alpha_eff_rad = np.radians(alpha_deg - alpha_0l_deg)

        stall_deg = np.clip(15.5 - 12.0 * max(m - 0.02, 0.0) + 18.0 * max(t - 0.12, 0.0), 10.0, 20.0)
        stall_rad = np.radians(stall_deg)

        cl_lin = cl_slope * alpha_eff_rad
        excess = np.maximum(0.0, np.abs(alpha_rad) - stall_rad)
        stall_factor = 1.0 / (1.0 + 3.5 * (excess ** 1.6))
        cl_total = cl_lin * stall_factor + 0.35 * np.sin(2.0 * alpha_rad) * (1.0 - stall_factor)

        cd0 = (0.0052 + 0.022 * t + 0.04 * (m ** 2)) * re_factor
        cd_induced = k_ind * (cl_total ** 2)
        stall_drag = 0.25 * (excess ** 1.5)
        cd_total = cd0 + cd_induced + stall_drag

        ld_ratio = cl_total / np.maximum(cd_total, 1e-4)

        # 2, 3, 4. Core Aerodynamic Curves
        ax_cl.plot(alpha_deg, cl_total, color=color, lw=2.0, label=code)
        ax_ld.plot(alpha_deg, ld_ratio, color=color, lw=2.0, label=code)
        ax_polar.plot(cd_total, cl_total, color=color, lw=2.0, label=code)

        # 5. Level-Flight Drag vs. Airspeed
        q_v = 0.5 * rho * (v_sweep ** 2)
        cl_req_v = weight_n / (q_v * s_ref)
        d_par_v = q_v * s_ref * cd0
        d_ind_v = q_v * s_ref * k_ind * (cl_req_v ** 2)
        d_tot_v = d_par_v + d_ind_v

        v_best_ld = float(v_sweep[int(np.argmin(d_tot_v))])
        min_drag_n = float(np.min(d_tot_v))

        ax_drag_v.plot(v_sweep, d_tot_v, color=color, lw=2.0, label=f"{code} (V_bg={v_best_ld:.1f} m/s)")
        ax_drag_v.scatter([v_best_ld], [min_drag_n], color=color, s=35, zorder=5)

        # 6. Glide Range from start_alt_m
        glide_range_km = (max(start_alt_m, 10.0) * np.maximum(ld_ratio, 0.0)) / 1000.0
        ax_glide.plot(alpha_deg, glide_range_km, color=color, lw=2.0, label=code)

        max_cl = float(np.max(cl_total))
        best_ld = float(np.max(ld_ratio))
        best_ld_aoa = float(alpha_deg[int(np.argmax(ld_ratio))])
        max_glide_km = float(np.max(glide_range_km))
        v_stall = float(np.sqrt((2.0 * weight_n) / (rho * s_ref * max(max_cl, 0.2))))

        report_lines.append(
            f"- **{code}:** Peak CL ≈ {max_cl:.2f} (V_stall ≈ {v_stall:.1f} m/s), "
            f"Best L/D ≈ {best_ld:.1f} at α ≈ {best_ld_aoa:.1f}°, "
            f"Best-Glide Speed V_bg ≈ {v_best_ld:.1f} m/s, Max Glide Range ≈ {max_glide_km:.2f} km."
        )

    # Styling Panel 1: Geometry
    ax_geom.axhline(0.0, color="#64748b", linestyle="--", lw=0.8)
    ax_geom.set_aspect("equal", adjustable="box")
    ax_geom.set_xlim(-0.05, 1.05)
    ax_geom.set_ylim(-0.35, 0.35)
    ax_geom.set_title("1. Airfoil Cross-Sections (x/c)", color="#f1f5f9", fontsize=10, pad=6)
    ax_geom.set_xlabel("Chordwise Position, x/c", color="#cbd5e1", fontsize=8.5)
    ax_geom.set_ylabel("y/c", color="#cbd5e1", fontsize=8.5)
    ax_geom.legend(loc="upper right", fontsize=7, facecolor="#1e293b", edgecolor="#475569", labelcolor="#f8fafc")

    # Styling Panel 2: CL vs AoA
    ax_cl.axhline(0.0, color="#64748b", linestyle=":", lw=0.8)
    ax_cl.axvline(0.0, color="#64748b", linestyle=":", lw=0.8)
    ax_cl.set_title(f"2. Lift Curve — CL vs. AoA (AR={aspect_ratio:.1f}, e={oswald_e:.2f})", color="#f1f5f9", fontsize=10, pad=6)
    ax_cl.set_xlabel("Angle of Attack, α (deg)", color="#cbd5e1", fontsize=8.5)
    ax_cl.set_ylabel("Lift Coefficient, CL", color="#cbd5e1", fontsize=8.5)
    ax_cl.legend(loc="upper left", fontsize=7.5, facecolor="#1e293b", edgecolor="#475569", labelcolor="#f8fafc")

    # Styling Panel 3: L/D vs AoA
    ax_ld.axhline(0.0, color="#64748b", linestyle=":", lw=0.8)
    ax_ld.set_title("3. Aerodynamic Efficiency — L/D vs. AoA", color="#f1f5f9", fontsize=10, pad=6)
    ax_ld.set_xlabel("Angle of Attack, α (deg)", color="#cbd5e1", fontsize=8.5)
    ax_ld.set_ylabel("Lift-to-Drag Ratio, L/D", color="#cbd5e1", fontsize=8.5)
    ax_ld.legend(loc="upper right", fontsize=7.5, facecolor="#1e293b", edgecolor="#475569", labelcolor="#f8fafc")

    # Styling Panel 4: Drag Polar
    ax_polar.set_title("4. Drag Polar — CL vs. CD", color="#f1f5f9", fontsize=10, pad=6)
    ax_polar.set_xlabel("Total Drag Coefficient, CD", color="#cbd5e1", fontsize=8.5)
    ax_polar.set_ylabel("Lift Coefficient, CL", color="#cbd5e1", fontsize=8.5)
    ax_polar.legend(loc="lower right", fontsize=7.5, facecolor="#1e293b", edgecolor="#475569", labelcolor="#f8fafc")

    # Styling Panel 5: Level-Flight Drag vs Airspeed
    ax_drag_v.axvline(velocity_ms, color="#f8fafc", linestyle="--", lw=1.0, alpha=0.75, label=f"Design Cruise ({velocity_ms:.0f} m/s)")
    ax_drag_v.set_title(f"5. Level-Flight Total Drag vs. Airspeed (m={mass_kg:.1f} kg, S={s_ref:.2f} m²)", color="#f1f5f9", fontsize=10, pad=6)
    ax_drag_v.set_xlabel("Airspeed, V (m/s)", color="#cbd5e1", fontsize=8.5)
    ax_drag_v.set_ylabel("Total Drag Force, D (N)", color="#cbd5e1", fontsize=8.5)
    ax_drag_v.set_ylim(bottom=0.0, top=max(weight_n * 0.45, 5.0))
    ax_drag_v.legend(loc="upper right", fontsize=7, facecolor="#1e293b", edgecolor="#475569", labelcolor="#f8fafc")

    # Styling Panel 6: Glide Range vs AoA
    ax_glide.set_title(f"6. Still-Air Glide Range from h0 = {start_alt_m:.0f} m", color="#f1f5f9", fontsize=10, pad=6)
    ax_glide.set_xlabel("Angle of Attack, α (deg)", color="#cbd5e1", fontsize=8.5)
    ax_glide.set_ylabel("Glide Distance (km)", color="#cbd5e1", fontsize=8.5)
    ax_glide.legend(loc="upper right", fontsize=7.5, facecolor="#1e293b", edgecolor="#475569", labelcolor="#f8fafc")

    fig.suptitle(
        f"Wing Profile Lift/Drag, Cruise Speed & Glide Trade-Off Report  |  AR={aspect_ratio:.1f}, S={s_ref:.2f} m², m={mass_kg:.1f} kg, h0={start_alt_m:.0f} m",
        color="#f8fafc", fontsize=11.5, fontweight="bold", y=0.985
    )
    plt.tight_layout(rect=[0, 0, 1, 0.965])

    tradeoff_md = (
        "### 📊 Design Trade-Off, Cruise & Glide Summary\n"
        + "\n".join(report_lines)
        + "\n\n**Key Engineering Trade-Offs:**\n"
        "- **High Camber (e.g., NACA 4412 / 6412):** Lowers stall speed (V_stall) and increases max glide distance "
        "from h0, but increases parasitic drag at high cruise airspeeds where induced drag drops.\n"
        "- **Symmetric / Low Camber (e.g., NACA 0012 / 2412):** Better suited for higher-speed cruise and "
        "symmetric/aerobatic maneuvering, at the cost of higher stall speed and shorter glide range.\n"
        "- **Best-Glide Speed (V_bg):** Occurs where induced drag equals parasitic drag (D_induced = D_parasitic), "
        "maximizing L/D and glide range."
    )

    return fig, tradeoff_md


with gr.Blocks() as demo:
    gr.Markdown("## ✈️ Wing Profile Lift/Drag, Cruise & Glide Visual Trade-Off Tool")

    with gr.Row():
        with gr.Column(scale=1):
            selected_airfoils = gr.CheckboxGroup(
                choices=AVAILABLE_AIRFOILS,
                value=["NACA 0012", "NACA 2412", "NACA 4412"],
                label="Select Wing / Airfoil Profiles"
            )
            alpha_min_deg = gr.Slider(-10.0, 4.0, value=-4.0, step=1.0, label="Min Angle of Attack (deg)")
            alpha_max_deg = gr.Slider(8.0, 26.0, value=18.0, step=1.0, label="Max Angle of Attack (deg)")
            aspect_ratio = gr.Slider(3.0, 16.0, value=8.0, step=0.5, label="Wing Aspect Ratio (AR)")
            oswald_e = gr.Slider(0.55, 0.95, value=0.80, step=0.02, label="Oswald Efficiency Factor (e)")
            wing_area_m2 = gr.Slider(0.10, 2.50, value=0.45, step=0.05, label="Wing Reference Area, S (m²)")
            mass_kg = gr.Slider(0.5, 25.0, value=3.5, step=0.5, label="Aircraft Mass, m (kg)")
            velocity_ms = gr.Slider(10.0, 65.0, value=22.0, step=1.0, label="Design Cruise Airspeed, V (m/s)")
            start_alt_m = gr.Slider(50.0, 2000.0, value=300.0, step=50.0, label="Initial Glide Altitude, h0 (m)")

            run_btn = gr.Button("Generate Visual Trade-Off Report", variant="primary")

        with gr.Column(scale=2):
            report_plot = gr.Plot(label="6-Panel Visual Trade-Off Report")
            report_text = gr.Markdown()

    inputs = [
        selected_airfoils, alpha_min_deg, alpha_max_deg,
        aspect_ratio, oswald_e, wing_area_m2, mass_kg, velocity_ms, start_alt_m
    ]
    run_btn.click(fn=simulate_airfoil_tradeoff, inputs=inputs, outputs=[report_plot, report_text])
    demo.load(fn=simulate_airfoil_tradeoff, inputs=inputs, outputs=[report_plot, report_text])

demo.launch(share=True, theme=gr.themes.Soft())

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4af2a6ba259877754c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
